In [1]:
import numpy as np
import spacy
import warnings
import os
import time
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import minimize
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import logging

# --- Together AI Client ---
from together import Together

# Suppress warnings for clean terminal output
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Sampler as LocalSampler

# --- Configuration ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define the old CSV to read from and the new CSV to write to
OLD_CSV_FILENAME = "qrag_telemetry_N150_run_1783611471_final.csv"  # <-- UPDATE THIS TO YOUR PREVIOUS RUN'S CSV
RUN_TIMESTAMP = int(time.time())
CSV_FILENAME = f"qrag_telemetry_Updated_run_{RUN_TIMESTAMP}.csv"

# Load environment variables
load_dotenv()

# ==============================================================================
# DATASET PLACEHOLDER (NEW AGENT-PATIENT INVERSION SENTENCES)
# ==============================================================================
NEW_DATABASE = [
  {"class":"Agent-Patient Inversion","text":"The steamed vegetables boiled the bamboo vegetable steamer.","query":"What is the active syntactic subject performing the action?","truth":"the steamed vegetables","conflict":"the bamboo vegetable steamer"},
  {"class":"Agent-Patient Inversion","text":"The cracked nuts snapped the heavy metal nutcracker.","query":"What is the active syntactic subject performing the action?","truth":"the cracked nuts","conflict":"the heavy metal nutcracker"},
  {"class":"Agent-Patient Inversion","text":"The cut glass scored the diamond-tipped glass cutter.","query":"What is the active syntactic subject performing the action?","truth":"the cut glass","conflict":"the diamond-tipped glass cutter"},
  {"class":"Agent-Patient Inversion","text":"The melted solder wicked the copper desoldering braid.","query":"What is the active syntactic subject performing the action?","truth":"the melted solder","conflict":"the copper desoldering braid"},
  {"class":"Agent-Patient Inversion","text":"The poured driveway floated the magnesium bull float.","query":"What is the active syntactic subject performing the action?","truth":"the poured driveway","conflict":"the magnesium bull float"},
  {"class":"Agent-Patient Inversion","text":"The secured drywall screwed the electric drywall gun.","query":"What is the active syntactic subject performing the action?","truth":"the secured drywall","conflict":"the electric drywall gun"},
  {"class":"Agent-Patient Inversion","text":"The incubated cells warmed the regulated cell incubator.","query":"What is the active syntactic subject performing the action?","truth":"the incubated cells","conflict":"the regulated cell incubator"},
  {"class":"Agent-Patient Inversion","text":"The trimmed hedge sheared the electric hedge trimmer.","query":"What is the active syntactic subject performing the action?","truth":"the trimmed hedge","conflict":"the electric hedge trimmer"},
  {"class":"Agent-Patient Inversion","text":"The dried towels spun the electric vented drum clothes dryer.","query":"What is the active syntactic subject performing the action?","truth":"the dried towels","conflict":"the electric vented drum clothes dryer"},
  {"class":"Agent-Patient Inversion","text":"The pureed soup blended the immersion hand blender.","query":"What is the active syntactic subject performing the action?","truth":"the pureed soup","conflict":"the immersion hand blender"},
  {"class":"Agent-Patient Inversion","text":"The sifted flour dusted the rotating flour sifter.","query":"What is the active syntactic subject performing the action?","truth":"the sifted flour","conflict":"the rotating flour sifter"},
  {"class":"Agent-Patient Inversion","text":"The portioned rice scooped the wooden rice paddle.","query":"What is the active syntactic subject performing the action?","truth":"the portioned rice","conflict":"the wooden rice paddle"},
  {"class":"Agent-Patient Inversion","text":"The riveted sheet popped the heavy manual pop riveter.","query":"What is the active syntactic subject performing the action?","truth":"the riveted sheet","conflict":"the heavy manual pop riveter"},
  {"class":"Agent-Patient Inversion","text":"The stripped paint peeled the high carbon paint scraper.","query":"What is the active syntactic subject performing the action?","truth":"the stripped paint","conflict":"the high carbon paint scraper"},
  {"class":"Agent-Patient Inversion","text":"The washed window cleared the rubber window squeegee.","query":"What is the active syntactic subject performing the action?","truth":"the washed window","conflict":"the rubber window squeegee"},
  {"class":"Agent-Patient Inversion","text":"The chopped brush shattered the heavy steel brush machete.","query":"What is the active syntactic subject performing the action?","truth":"the chopped brush","conflict":"the heavy steel brush machete"},
  {"class":"Agent-Patient Inversion","text":"The purified water filtered the ceramic pump backpacking water filter.","query":"What is the active syntactic subject performing the action?","truth":"the purified water","conflict":"the ceramic pump backpacking water filter"},
  {"class":"Agent-Patient Inversion","text":"The drained bilge pumped the submersible automatic bilge pump.","query":"What is the active syntactic subject performing the action?","truth":"the drained bilge","conflict":"the submersible automatic bilge pump"},
  {"class":"Agent-Patient Inversion","text":"The plowed snow pushed the angled steel snow plow blade.","query":"What is the active syntactic subject performing the action?","truth":"the plowed snow","conflict":"the angled steel snow plow blade"},
  {"class":"Agent-Patient Inversion","text":"The brushed teeth cleaned the oscillating electric toothbrush.","query":"What is the active syntactic subject performing the action?","truth":"the brushed teeth","conflict":"the oscillating electric toothbrush"},
  {"class":"Agent-Patient Inversion","text":"The frozen ice cracked the plastic square ice mold.","query":"What is the active syntactic subject performing the action?","truth":"the frozen ice","conflict":"the plastic square ice mold"},
  {"class":"Agent-Patient Inversion","text":"The burnt wood charred the iron outdoor fire pit.","query":"What is the active syntactic subject performing the action?","truth":"the burnt wood","conflict":"the iron outdoor fire pit"},
  {"class":"Agent-Patient Inversion","text":"The dissolved powder melted the magnetic liquid stir bar.","query":"What is the active syntactic subject performing the action?","truth":"the dissolved powder","conflict":"the magnetic liquid stir bar"},
  {"class":"Agent-Patient Inversion","text":"The stretched rubber snapped the steel mechanical tensioners.","query":"What is the active syntactic subject performing the action?","truth":"the stretched rubber","conflict":"the steel mechanical tensioners"},
  {"class":"Agent-Patient Inversion","text":"The cooked pasta boiled the enameled cast iron pot.","query":"What is the active syntactic subject performing the action?","truth":"the cooked pasta","conflict":"the enameled cast iron pot"},
  {"class":"Agent-Patient Inversion","text":"The compressed gas expanded the heavy brass regulator valve.","query":"What is the active syntactic subject performing the action?","truth":"the compressed gas","conflict":"the heavy brass regulator valve"},
  {"class":"Agent-Patient Inversion","text":"The deflated balloon shrunk the motorized air compressor.","query":"What is the active syntactic subject performing the action?","truth":"the deflated balloon","conflict":"the motorized air compressor"},
  {"class":"Agent-Patient Inversion","text":"The spilled coffee stained the porous ceramic desktop coaster.","query":"What is the active syntactic subject performing the action?","truth":"the spilled coffee","conflict":"the porous ceramic desktop coaster"},
  {"class":"Agent-Patient Inversion","text":"The heated metal warped the industrial induction heating coil.","query":"What is the active syntactic subject performing the action?","truth":"the heated metal","conflict":"the industrial induction heating coil"},
  {"class":"Agent-Patient Inversion","text":"The chilled wort cooled the copper immersion liquid chiller.","query":"What is the active syntactic subject performing the action?","truth":"the chilled wort","conflict":"the copper immersion liquid chiller"},
  {"class":"Agent-Patient Inversion","text":"The torn document ripped the motorized paper shredding machine.","query":"What is the active syntactic subject performing the action?","truth":"the torn document","conflict":"the motorized paper shredding machine"},
  {"class":"Agent-Patient Inversion","text":"The dropped vase shattered the thick rubber floor mat.","query":"What is the active syntactic subject performing the action?","truth":"the dropped vase","conflict":"the thick rubber floor mat"},
  {"class":"Agent-Patient Inversion","text":"The flipped pancake turned the flat nylon cooking spatula.","query":"What is the active syntactic subject performing the action?","truth":"the flipped pancake","conflict":"the flat nylon cooking spatula"},
  {"class":"Agent-Patient Inversion","text":"The shifted gravel rolled the hydraulic heavy dump truck.","query":"What is the active syntactic subject performing the action?","truth":"the shifted gravel","conflict":"the hydraulic heavy dump truck"},
  {"class":"Agent-Patient Inversion","text":"The blocked pipe burst the pressurized copper water valve.","query":"What is the active syntactic subject performing the action?","truth":"the blocked pipe","conflict":"the pressurized copper water valve"},
  {"class":"Agent-Patient Inversion","text":"The condensed steam dripped the stainless commercial vent hood.","query":"What is the active syntactic subject performing the action?","truth":"the condensed steam","conflict":"the stainless commercial vent hood"},
  {"class":"Agent-Patient Inversion","text":"The mixed dough kneaded the heavy planetary stand mixer.","query":"What is the active syntactic subject performing the action?","truth":"the mixed dough","conflict":"the heavy planetary stand mixer"},
  {"class":"Agent-Patient Inversion","text":"The filtered sunlight dimmed the polarized glass window pane.","query":"What is the active syntactic subject performing the action?","truth":"the filtered sunlight","conflict":"the polarized glass window pane"},
  {"class":"Agent-Patient Inversion","text":"The rusted bolt broke the hardened steel socket wrench.","query":"What is the active syntactic subject performing the action?","truth":"the rusted bolt","conflict":"the hardened steel socket wrench"},
  {"class":"Agent-Patient Inversion","text":"The carved turkey sliced the motorized electric carving knife.","query":"What is the active syntactic subject performing the action?","truth":"the carved turkey","conflict":"the motorized electric carving knife"},
  {"class":"Agent-Patient Inversion","text":"The whipped cream stiffened the wire steel balloon whisk.","query":"What is the active syntactic subject performing the action?","truth":"the whipped cream","conflict":"the wire steel balloon whisk"},
  {"class":"Agent-Patient Inversion","text":"The grated cheese shredded the rotary stainless steel grater.","query":"What is the active syntactic subject performing the action?","truth":"the grated cheese","conflict":"the rotary stainless steel grater"},
  {"class":"Agent-Patient Inversion","text":"The crushed ice smashed the heavy wooden cocktail muddler.","query":"What is the active syntactic subject performing the action?","truth":"the crushed ice","conflict":"the heavy wooden cocktail muddler"},
  {"class":"Agent-Patient Inversion","text":"The baked bread crusted the hot ceramic baking stone.","query":"What is the active syntactic subject performing the action?","truth":"the baked bread","conflict":"the hot ceramic baking stone"},
  {"class":"Agent-Patient Inversion","text":"The sliced apple split the spring-loaded mechanical apple corer.","query":"What is the active syntactic subject performing the action?","truth":"the sliced apple","conflict":"the spring-loaded mechanical apple corer"},
  {"class":"Agent-Patient Inversion","text":"The peeled potato skinned the swiveling blade vegetable peeler.","query":"What is the active syntactic subject performing the action?","truth":"the peeled potato","conflict":"the swiveling blade vegetable peeler"},
  {"class":"Agent-Patient Inversion","text":"The mashed avocado squashed the flat metal avocado masher.","query":"What is the active syntactic subject performing the action?","truth":"the mashed avocado","conflict":"the flat metal avocado masher"},
  {"class":"Agent-Patient Inversion","text":"The drained pasta emptied the perforated aluminum kitchen colander.","query":"What is the active syntactic subject performing the action?","truth":"the drained pasta","conflict":"the perforated aluminum kitchen colander"},
  {"class":"Agent-Patient Inversion","text":"The toasted bagel browned the four-slot electric bread toaster.","query":"What is the active syntactic subject performing the action?","truth":"the toasted bagel","conflict":"the four-slot electric bread toaster"},
  {"class":"Agent-Patient Inversion","text":"The fried egg cooked the seasoned cast iron skillet.","query":"What is the active syntactic subject performing the action?","truth":"the fried egg","conflict":"the seasoned cast iron skillet"},
  {"class":"Agent-Patient Inversion","text":"The brewed coffee percolated the programmable drip coffee maker.","query":"What is the active syntactic subject performing the action?","truth":"the brewed coffee","conflict":"the programmable drip coffee maker"},
  {"class":"Agent-Patient Inversion","text":"The poured wine aerated the wide crystal wine decanter.","query":"What is the active syntactic subject performing the action?","truth":"the poured wine","conflict":"the wide crystal wine decanter"},
  {"class":"Agent-Patient Inversion","text":"The spun salad dried the plastic rotating salad spinner.","query":"What is the active syntactic subject performing the action?","truth":"the spun salad","conflict":"the plastic rotating salad spinner"},
  {"class":"Agent-Patient Inversion","text":"The mixed cocktail chilled the stainless steel cocktail shaker.","query":"What is the active syntactic subject performing the action?","truth":"the mixed cocktail","conflict":"the stainless steel cocktail shaker"},
  {"class":"Agent-Patient Inversion","text":"The seasoned steak marinated the flexible silicone basting brush.","query":"What is the active syntactic subject performing the action?","truth":"the seasoned steak","conflict":"the flexible silicone basting brush"},
  {"class":"Agent-Patient Inversion","text":"The opened can punctured the heavy-duty manual can opener.","query":"What is the active syntactic subject performing the action?","truth":"the opened can","conflict":"the heavy-duty manual can opener"},
  {"class":"Agent-Patient Inversion","text":"The uncorked bottle popped the double-hinged sommelier wine key.","query":"What is the active syntactic subject performing the action?","truth":"the uncorked bottle","conflict":"the double-hinged sommelier wine key"},
  {"class":"Agent-Patient Inversion","text":"The measured flour leveled the graduated plastic measuring cup.","query":"What is the active syntactic subject performing the action?","truth":"the measured flour","conflict":"the graduated plastic measuring cup"},
  {"class":"Agent-Patient Inversion","text":"The rolled dough flattened the heavy marble rolling pin.","query":"What is the active syntactic subject performing the action?","truth":"the rolled dough","conflict":"the heavy marble rolling pin"},
  {"class":"Agent-Patient Inversion","text":"The chopped onion diced the stainless steel chef knife.","query":"What is the active syntactic subject performing the action?","truth":"the chopped onion","conflict":"the stainless steel chef knife"},
  {"class":"Agent-Patient Inversion","text":"The minced garlic squeezed the heavy metal garlic press.","query":"What is the active syntactic subject performing the action?","truth":"the minced garlic","conflict":"the heavy metal garlic press"},
  {"class":"Agent-Patient Inversion","text":"The tenderized meat flattened the spiked aluminum meat mallet.","query":"What is the active syntactic subject performing the action?","truth":"the tenderized meat","conflict":"the spiked aluminum meat mallet"},
  {"class":"Agent-Patient Inversion","text":"The scooped ice cream formed the antifreeze aluminum ice cream scoop.","query":"What is the active syntactic subject performing the action?","truth":"the scooped ice cream","conflict":"the antifreeze aluminum ice cream scoop"},
  {"class":"Agent-Patient Inversion","text":"The juiced lemon squirted the motorized citrus juicing machine.","query":"What is the active syntactic subject performing the action?","truth":"the juiced lemon","conflict":"the motorized citrus juicing machine"},
  {"class":"Agent-Patient Inversion","text":"The strained broth filtered the fine wire mesh sieve.","query":"What is the active syntactic subject performing the action?","truth":"the strained broth","conflict":"the fine wire mesh sieve"},
  {"class":"Agent-Patient Inversion","text":"The carved wood splintered the high-speed rotary carving tool.","query":"What is the active syntactic subject performing the action?","truth":"the carved wood","conflict":"the high-speed rotary carving tool"},
  {"class":"Agent-Patient Inversion","text":"The sanded floor smoothed the heavy orbital floor sander.","query":"What is the active syntactic subject performing the action?","truth":"the sanded floor","conflict":"the heavy orbital floor sander"},
  {"class":"Agent-Patient Inversion","text":"The planed board shaved the cast-iron bench hand plane.","query":"What is the active syntactic subject performing the action?","truth":"the planed board","conflict":"the cast-iron bench hand plane"},
  {"class":"Agent-Patient Inversion","text":"The drilled hole bored the titanium-coated twist drill bit.","query":"What is the active syntactic subject performing the action?","truth":"the drilled hole","conflict":"the titanium-coated twist drill bit"},
  {"class":"Agent-Patient Inversion","text":"The sawed timber ripped the carbide-tipped circular saw blade.","query":"What is the active syntactic subject performing the action?","truth":"the sawed timber","conflict":"the carbide-tipped circular saw blade"},
  {"class":"Agent-Patient Inversion","text":"The routed edge shaped the variable-speed plunge wood router.","query":"What is the active syntactic subject performing the action?","truth":"the routed edge","conflict":"the variable-speed plunge wood router"},
  {"class":"Agent-Patient Inversion","text":"The chiseled mortise split the bevel-edge steel wood chisel.","query":"What is the active syntactic subject performing the action?","truth":"the chiseled mortise","conflict":"the bevel-edge steel wood chisel"},
  {"class":"Agent-Patient Inversion","text":"The glued joint bonded the adjustable steel bar clamp.","query":"What is the active syntactic subject performing the action?","truth":"the glued joint","conflict":"the adjustable steel bar clamp"},
  {"class":"Agent-Patient Inversion","text":"The bolted frame tightened the heavy pneumatic impact wrench.","query":"What is the active syntactic subject performing the action?","truth":"the bolted frame","conflict":"the heavy pneumatic impact wrench"},
  {"class":"Agent-Patient Inversion","text":"The welded seam fused the portable MIG wire welder.","query":"What is the active syntactic subject performing the action?","truth":"the welded seam","conflict":"the portable MIG wire welder"},
  {"class":"Agent-Patient Inversion","text":"The soldered wire melted the fine-tipped butane soldering iron.","query":"What is the active syntactic subject performing the action?","truth":"the soldered wire","conflict":"the fine-tipped butane soldering iron"},
  {"class":"Agent-Patient Inversion","text":"The filed metal scraped the coarse bastard cut hand file.","query":"What is the active syntactic subject performing the action?","truth":"the filed metal","conflict":"the coarse bastard cut hand file"},
  {"class":"Agent-Patient Inversion","text":"The hammered nail drove the forged steel claw hammer.","query":"What is the active syntactic subject performing the action?","truth":"the hammered nail","conflict":"the forged steel claw hammer"},
  {"class":"Agent-Patient Inversion","text":"The pried board lifted the hexagonal steel crowbar pry bar.","query":"What is the active syntactic subject performing the action?","truth":"the pried board","conflict":"the hexagonal steel crowbar pry bar"},
  {"class":"Agent-Patient Inversion","text":"The tightened pipe gripped the heavy adjustable pipe wrench.","query":"What is the active syntactic subject performing the action?","truth":"the tightened pipe","conflict":"the heavy adjustable pipe wrench"},
  {"class":"Agent-Patient Inversion","text":"The stripped wire exposed the spring-loaded automatic wire stripper.","query":"What is the active syntactic subject performing the action?","truth":"the stripped wire","conflict":"the spring-loaded automatic wire stripper"},
  {"class":"Agent-Patient Inversion","text":"The crimped terminal squeezed the heavy duty ratcheting wire crimper.","query":"What is the active syntactic subject performing the action?","truth":"the crimped terminal","conflict":"the heavy duty ratcheting wire crimper"},
  {"class":"Agent-Patient Inversion","text":"The cut cable snapped the high-leverage diagonal wire cutters.","query":"What is the active syntactic subject performing the action?","truth":"the cut cable","conflict":"the high-leverage diagonal wire cutters"},
  {"class":"Agent-Patient Inversion","text":"The taped joint sealed the flexible black electrical tape.","query":"What is the active syntactic subject performing the action?","truth":"the taped joint","conflict":"the flexible black electrical tape"},
  {"class":"Agent-Patient Inversion","text":"The stapled wire fastened the heavy duty manual staple gun.","query":"What is the active syntactic subject performing the action?","truth":"the stapled wire","conflict":"the heavy duty manual staple gun"},
  {"class":"Agent-Patient Inversion","text":"The measured board spanned the retractable steel tape measure.","query":"What is the active syntactic subject performing the action?","truth":"the measured board","conflict":"the retractable steel tape measure"},
  {"class":"Agent-Patient Inversion","text":"The leveled beam balanced the anodized aluminum bubble level.","query":"What is the active syntactic subject performing the action?","truth":"the leveled beam","conflict":"the anodized aluminum bubble level"},
  {"class":"Agent-Patient Inversion","text":"The marked line snapped the powdered blue chalk line.","query":"What is the active syntactic subject performing the action?","truth":"the marked line","conflict":"the powdered blue chalk line"},
  {"class":"Agent-Patient Inversion","text":"The squared corner aligned the aluminum framing carpenter square.","query":"What is the active syntactic subject performing the action?","truth":"the squared corner","conflict":"the aluminum framing carpenter square"},
  {"class":"Agent-Patient Inversion","text":"The painted wall brushed the synthetic angled sash brush.","query":"What is the active syntactic subject performing the action?","truth":"the painted wall","conflict":"the synthetic angled sash brush"},
  {"class":"Agent-Patient Inversion","text":"The rolled ceiling coated the fuzzy microfiber paint roller.","query":"What is the active syntactic subject performing the action?","truth":"the rolled ceiling","conflict":"the fuzzy microfiber paint roller"},
  {"class":"Agent-Patient Inversion","text":"The caulked gap filled the dripless skeletal caulk gun.","query":"What is the active syntactic subject performing the action?","truth":"the caulked gap","conflict":"the dripless skeletal caulk gun"},
  {"class":"Agent-Patient Inversion","text":"The plastered seam smoothed the flexible steel joint taping knife.","query":"What is the active syntactic subject performing the action?","truth":"the plastered seam","conflict":"the flexible steel joint taping knife"},
  {"class":"Agent-Patient Inversion","text":"The troweled concrete flattened the magnesium hand masonry trowel.","query":"What is the active syntactic subject performing the action?","truth":"the troweled concrete","conflict":"the magnesium hand masonry trowel"},
  {"class":"Agent-Patient Inversion","text":"The dug hole opened the pointed steel digging shovel.","query":"What is the active syntactic subject performing the action?","truth":"the dug hole","conflict":"the pointed steel digging shovel"},
  {"class":"Agent-Patient Inversion","text":"The raked leaves gathered the flexible plastic leaf rake.","query":"What is the active syntactic subject performing the action?","truth":"the raked leaves","conflict":"the flexible plastic leaf rake"},
  {"class":"Agent-Patient Inversion","text":"The swept floor cleared the wide synthetic push broom.","query":"What is the active syntactic subject performing the action?","truth":"the swept floor","conflict":"the wide synthetic push broom"},
  {"class":"Agent-Patient Inversion","text":"The mopped spill dried the absorbent cotton loop mop.","query":"What is the active syntactic subject performing the action?","truth":"the mopped spill","conflict":"the absorbent cotton loop mop"},
  {"class":"Agent-Patient Inversion","text":"The vacuumed dust emptied the high-suction wet dry vacuum.","query":"What is the active syntactic subject performing the action?","truth":"the vacuumed dust","conflict":"the high-suction wet dry vacuum"},
  {"class":"Agent-Patient Inversion","text":"The polished brass shined the soft rotary buffing wheel.","query":"What is the active syntactic subject performing the action?","truth":"the polished brass","conflict":"the soft rotary buffing wheel"},
  {"class":"Agent-Patient Inversion","text":"The engraved tag scratched the pneumatic vibrating engraving pen.","query":"What is the active syntactic subject performing the action?","truth":"the engraved tag","conflict":"the pneumatic vibrating engraving pen"},
  {"class":"Agent-Patient Inversion","text":"The mixed mortar churned the portable electric cement mixer.","query":"What is the active syntactic subject performing the action?","truth":"the mixed mortar","conflict":"the portable electric cement mixer"},
  {"class":"Agent-Patient Inversion","text":"The compacted dirt settled the heavy steel hand tamper.","query":"What is the active syntactic subject performing the action?","truth":"the compacted dirt","conflict":"the heavy steel hand tamper"},
  {"class":"Agent-Patient Inversion","text":"The hoisted engine lifted the hydraulic rolling engine crane.","query":"What is the active syntactic subject performing the action?","truth":"the hoisted engine","conflict":"the hydraulic rolling engine crane"},
  {"class":"Agent-Patient Inversion","text":"The jacked chassis raised the low-profile hydraulic floor jack.","query":"What is the active syntactic subject performing the action?","truth":"the jacked chassis","conflict":"the low-profile hydraulic floor jack"},
  {"class":"Agent-Patient Inversion","text":"The inflated tire expanded the twin-cylinder stationary air compressor.","query":"What is the active syntactic subject performing the action?","truth":"the inflated tire","conflict":"the twin-cylinder stationary air compressor"},
  {"class":"Agent-Patient Inversion","text":"The lubricated gear oiled the precision needle oiler bottle.","query":"What is the active syntactic subject performing the action?","truth":"the lubricated gear","conflict":"the precision needle oiler bottle"},
  {"class":"Agent-Patient Inversion","text":"The greased fitting filled the lever-action pneumatic grease gun.","query":"What is the active syntactic subject performing the action?","truth":"the greased fitting","conflict":"the lever-action pneumatic grease gun"},
  {"class":"Agent-Patient Inversion","text":"The jumped battery sparked the heavy-gauge copper jumper cables.","query":"What is the active syntactic subject performing the action?","truth":"the jumped battery","conflict":"the heavy-gauge copper jumper cables"},
  {"class":"Agent-Patient Inversion","text":"The washed car shined the soft foaming automotive wash mitt.","query":"What is the active syntactic subject performing the action?","truth":"the washed car","conflict":"the soft foaming automotive wash mitt"},
  {"class":"Agent-Patient Inversion","text":"The waxed paint gleamed the dual-action random orbital polisher.","query":"What is the active syntactic subject performing the action?","truth":"the waxed paint","conflict":"the dual-action random orbital polisher"},
  {"class":"Agent-Patient Inversion","text":"The towed trailer pulled the forged steel class III trailer hitch.","query":"What is the active syntactic subject performing the action?","truth":"the towed trailer","conflict":"the forged steel class III trailer hitch"},
  {"class":"Agent-Patient Inversion","text":"The secured load strapped the heavy-duty nylon ratchet tie-down.","query":"What is the active syntactic subject performing the action?","truth":"the secured load","conflict":"the heavy-duty nylon ratchet tie-down"},
  {"class":"Agent-Patient Inversion","text":"The steered vehicle turned the leather-wrapped steering wheel.","query":"What is the active syntactic subject performing the action?","truth":"the steered vehicle","conflict":"the leather-wrapped steering wheel"},
  {"class":"Agent-Patient Inversion","text":"The braked rotor stopped the semi-metallic disc brake pad.","query":"What is the active syntactic subject performing the action?","truth":"the braked rotor","conflict":"the semi-metallic disc brake pad"},
  {"class":"Agent-Patient Inversion","text":"The accelerated engine roared the electronic fuel injection throttle.","query":"What is the active syntactic subject performing the action?","truth":"the accelerated engine","conflict":"the electronic fuel injection throttle"},
  {"class":"Agent-Patient Inversion","text":"The shifted gear changed the short-throw manual transmission stick.","query":"What is the active syntactic subject performing the action?","truth":"the shifted gear","conflict":"the short-throw manual transmission stick"},
  {"class":"Agent-Patient Inversion","text":"The wiped windshield cleared the articulated rubber wiper blade.","query":"What is the active syntactic subject performing the action?","truth":"the wiped windshield","conflict":"the articulated rubber wiper blade"},
  {"class":"Agent-Patient Inversion","text":"The honked horn blared the steering wheel center airbag pad.","query":"What is the active syntactic subject performing the action?","truth":"the honked horn","conflict":"the steering wheel center airbag pad"},
  {"class":"Agent-Patient Inversion","text":"The fueled tank filled the high-flow unleaded gasoline nozzle.","query":"What is the active syntactic subject performing the action?","truth":"the fueled tank","conflict":"the high-flow unleaded gasoline nozzle"},
  {"class":"Agent-Patient Inversion","text":"The charged battery powered the rapid direct current supercharger.","query":"What is the active syntactic subject performing the action?","truth":"the charged battery","conflict":"the rapid direct current supercharger"},
  {"class":"Agent-Patient Inversion","text":"The locked door clicked the remote keyless entry key fob.","query":"What is the active syntactic subject performing the action?","truth":"the locked door","conflict":"the remote keyless entry key fob"},
  {"class":"Agent-Patient Inversion","text":"The opened window dropped the electric power window regulator switch.","query":"What is the active syntactic subject performing the action?","truth":"the opened window","conflict":"the electric power window regulator switch"},
  {"class":"Agent-Patient Inversion","text":"The adjusted mirror turned the motorized side view mirror control.","query":"What is the active syntactic subject performing the action?","truth":"the adjusted mirror","conflict":"the motorized side view mirror control"},
  {"class":"Agent-Patient Inversion","text":"The heated seat warmed the integrated wire seat heating element.","query":"What is the active syntactic subject performing the action?","truth":"the heated seat","conflict":"the integrated wire seat heating element"},
  {"class":"Agent-Patient Inversion","text":"The cooled cabin chilled the engine-driven air conditioning compressor.","query":"What is the active syntactic subject performing the action?","truth":"the cooled cabin","conflict":"the engine-driven air conditioning compressor"},
  {"class":"Agent-Patient Inversion","text":"The exhausted fumes vented the stainless steel performance exhaust muffler.","query":"What is the active syntactic subject performing the action?","truth":"the exhausted fumes","conflict":"the stainless steel performance exhaust muffler"},
  {"class":"Agent-Patient Inversion","text":"The illuminated road brightened the high-intensity discharge projector headlight.","query":"What is the active syntactic subject performing the action?","truth":"the illuminated road","conflict":"the high-intensity discharge projector headlight"},
  {"class":"Agent-Patient Inversion","text":"The pedaled bicycle turned the forged aluminum bicycle crankset.","query":"What is the active syntactic subject performing the action?","truth":"the pedaled bicycle","conflict":"the forged aluminum bicycle crankset"},
  {"class":"Agent-Patient Inversion","text":"The shifted chain moved the indexed rear bicycle derailleur.","query":"What is the active syntactic subject performing the action?","truth":"the shifted chain","conflict":"the indexed rear bicycle derailleur"},
  {"class":"Agent-Patient Inversion","text":"The braked rim stopped the hydraulic bicycle rim brake caliper.","query":"What is the active syntactic subject performing the action?","truth":"the braked rim","conflict":"the hydraulic bicycle rim brake caliper"},
  {"class":"Agent-Patient Inversion","text":"The steered fork turned the carbon fiber drop handlebar.","query":"What is the active syntactic subject performing the action?","truth":"the steered fork","conflict":"the carbon fiber drop handlebar"},
  {"class":"Agent-Patient Inversion","text":"The inflated inner tube expanded the high-pressure track floor pump.","query":"What is the active syntactic subject performing the action?","truth":"the inflated inner tube","conflict":"the high-pressure track floor pump"},
  {"class":"Agent-Patient Inversion","text":"The patched puncture sealed the vulcanizing rubber tire patch kit.","query":"What is the active syntactic subject performing the action?","truth":"the patched puncture","conflict":"the vulcanizing rubber tire patch kit"},
  {"class":"Agent-Patient Inversion","text":"The locked frame secured the hardened steel bicycle U-lock.","query":"What is the active syntactic subject performing the action?","truth":"the locked frame","conflict":"the hardened steel bicycle U-lock"},
  {"class":"Agent-Patient Inversion","text":"The illuminated trail flashed the rechargeable LED bike headlight.","query":"What is the active syntactic subject performing the action?","truth":"the illuminated trail","conflict":"the rechargeable LED bike headlight"},
  {"class":"Agent-Patient Inversion","text":"The rowed skiff paddled the long varnished wooden oar.","query":"What is the active syntactic subject performing the action?","truth":"the rowed skiff","conflict":"the long varnished wooden oar"},
  {"class":"Agent-Patient Inversion","text":"The sailed dinghy caught the wind-filled triangular canvas jib.","query":"What is the active syntactic subject performing the action?","truth":"the sailed dinghy","conflict":"the wind-filled triangular canvas jib"},
  {"class":"Agent-Patient Inversion","text":"The steered yacht turned the brass-bound wooden ship wheel.","query":"What is the active syntactic subject performing the action?","truth":"the steered yacht","conflict":"the brass-bound wooden ship wheel"},
  {"class":"Agent-Patient Inversion","text":"The anchored schooner dropped the galvanized heavy plow anchor.","query":"What is the active syntactic subject performing the action?","truth":"the anchored schooner","conflict":"the galvanized heavy plow anchor"},
  {"class":"Agent-Patient Inversion","text":"The moored vessel tied the braided nylon dock mooring line.","query":"What is the active syntactic subject performing the action?","truth":"the moored vessel","conflict":"the braided nylon dock mooring line"},
  {"class":"Agent-Patient Inversion","text":"The navigated sea charted the dashboard marine GPS chartplotter.","query":"What is the active syntactic subject performing the action?","truth":"the navigated sea","conflict":"the dashboard marine GPS chartplotter"},
  {"class":"Agent-Patient Inversion","text":"The sounded depth beeped the acoustic hull-mounted depth sounder.","query":"What is the active syntactic subject performing the action?","truth":"the sounded depth","conflict":"the acoustic hull-mounted depth sounder"},
  {"class":"Agent-Patient Inversion","text":"The flown aircraft soared the aluminum wing trailing edge flap.","query":"What is the active syntactic subject performing the action?","truth":"the flown aircraft","conflict":"the aluminum wing trailing edge flap"},
  {"class":"Agent-Patient Inversion","text":"The steered jet turned the pilot control cockpit flight yoke.","query":"What is the active syntactic subject performing the action?","truth":"the steered jet","conflict":"the pilot control cockpit flight yoke"},
  {"class":"Agent-Patient Inversion","text":"The braked plane stopped the heavy pneumatic aircraft landing gear.","query":"What is the active syntactic subject performing the action?","truth":"the braked plane","conflict":"the heavy pneumatic aircraft landing gear"},
  {"class":"Agent-Patient Inversion","text":"The accelerated airliner roared the twin-spool turbofan jet engine.","query":"What is the active syntactic subject performing the action?","truth":"the accelerated airliner","conflict":"the twin-spool turbofan jet engine"},
  {"class":"Agent-Patient Inversion","text":"The navigated airspace plotted the integrated glass cockpit avionics suite.","query":"What is the active syntactic subject performing the action?","truth":"the navigated airspace","conflict":"the integrated glass cockpit avionics suite"},
  {"class":"Agent-Patient Inversion","text":"The hoisted container lifted the industrial heavy gantry crane.","query":"What is the active syntactic subject performing the action?","truth":"the hoisted container","conflict":"the industrial heavy gantry crane"},
  {"class":"Agent-Patient Inversion","text":"The moved pallet rolled the manual hydraulic pallet jack truck.","query":"What is the active syntactic subject performing the action?","truth":"the moved pallet","conflict":"the manual hydraulic pallet jack truck"},
  {"class":"Agent-Patient Inversion","text":"The transported inventory drove the heavy diesel counterbalanced forklift.","query":"What is the active syntactic subject performing the action?","truth":"the transported inventory","conflict":"the heavy diesel counterbalanced forklift"},
  {"class":"Agent-Patient Inversion","text":"The dumped payload tipped the hydraulic dump truck bed cylinder.","query":"What is the active syntactic subject performing the action?","truth":"the dumped payload","conflict":"the hydraulic dump truck bed cylinder"},
  {"class":"Agent-Patient Inversion","text":"The dug foundation scraped the hydraulic crawler excavator toothed bucket.","query":"What is the active syntactic subject performing the action?","truth":"the dug foundation","conflict":"the hydraulic crawler excavator toothed bucket"},
  {"class":"Agent-Patient Inversion","text":"The graded dirt leveled the heavy diesel motor grader blade.","query":"What is the active syntactic subject performing the action?","truth":"the graded dirt","conflict":"the heavy diesel motor grader blade"},
  {"class":"Agent-Patient Inversion","text":"The paved road rolled the vibratory steel drum asphalt roller.","query":"What is the active syntactic subject performing the action?","truth":"the paved road","conflict":"the vibratory steel drum asphalt roller"},
  {"class":"Agent-Patient Inversion","text":"The sanded ice scattered the automated truck-mounted salt spreader.","query":"What is the active syntactic subject performing the action?","truth":"the sanded ice","conflict":"the automated truck-mounted salt spreader"},
  {"class":"Agent-Patient Inversion","text":"The injected vaccine pierced the sterile hypodermic syringe needle.","query":"What is the active syntactic subject performing the action?","truth":"the injected vaccine","conflict":"the sterile hypodermic syringe needle"},
  {"class":"Agent-Patient Inversion","text":"The drawn blood filled the vacuum-sealed blood collection tube.","query":"What is the active syntactic subject performing the action?","truth":"the drawn blood","conflict":"the vacuum-sealed blood collection tube"},
  {"class":"Agent-Patient Inversion","text":"The stitched laceration closed the curved surgical suture needle.","query":"What is the active syntactic subject performing the action?","truth":"the stitched laceration","conflict":"the curved surgical suture needle"},
  {"class":"Agent-Patient Inversion","text":"The bandaged wound wrapped the sterile self-adhering gauze bandage.","query":"What is the active syntactic subject performing the action?","truth":"the bandaged wound","conflict":"the sterile self-adhering gauze bandage"},
  {"class":"Agent-Patient Inversion","text":"The checked pulse thumped the acoustic cardiology stethoscope chestpiece.","query":"What is the active syntactic subject performing the action?","truth":"the checked pulse","conflict":"the acoustic cardiology stethoscope chestpiece"},
  {"class":"Agent-Patient Inversion","text":"The measured temperature beeped the digital infrared ear thermometer.","query":"What is the active syntactic subject performing the action?","truth":"the measured temperature","conflict":"the digital infrared ear thermometer"},
  {"class":"Agent-Patient Inversion","text":"The weighed infant balanced the digital pediatric medical scale.","query":"What is the active syntactic subject performing the action?","truth":"the weighed infant","conflict":"the digital pediatric medical scale"},
  {"class":"Agent-Patient Inversion","text":"The spun plasma separated the high-speed laboratory centrifuge rotor.","query":"What is the active syntactic subject performing the action?","truth":"the spun plasma","conflict":"the high-speed laboratory centrifuge rotor"},
  {"class":"Agent-Patient Inversion","text":"The transferred reagent dropped the adjustable volume mechanical micropipette.","query":"What is the active syntactic subject performing the action?","truth":"the transferred reagent","conflict":"the adjustable volume mechanical micropipette"},
  {"class":"Agent-Patient Inversion","text":"The magnified bacteria focused the binocular compound light microscope.","query":"What is the active syntactic subject performing the action?","truth":"the magnified bacteria","conflict":"the binocular compound light microscope"},
  {"class":"Agent-Patient Inversion","text":"The scanned fracture radiated the digital medical x-ray machine.","query":"What is the active syntactic subject performing the action?","truth":"the scanned fracture","conflict":"the digital medical x-ray machine"},
  {"class":"Agent-Patient Inversion","text":"The monitored rhythm beeped the electrical electrocardiogram monitor machine.","query":"What is the active syntactic subject performing the action?","truth":"the monitored rhythm","conflict":"the electrical electrocardiogram monitor machine"},
  {"class":"Agent-Patient Inversion","text":"The oxygenated blood pumped the extracorporeal membrane oxygenation machine.","query":"What is the active syntactic subject performing the action?","truth":"the oxygenated blood","conflict":"the extracorporeal membrane oxygenation machine"},
  {"class":"Agent-Patient Inversion","text":"The ventilated lung breathed the mechanical intensive care ventilator.","query":"What is the active syntactic subject performing the action?","truth":"the ventilated lung","conflict":"the mechanical intensive care ventilator"},
  {"class":"Agent-Patient Inversion","text":"The shocked heart jumped the portable automated external defibrillator.","query":"What is the active syntactic subject performing the action?","truth":"the shocked heart","conflict":"the portable automated external defibrillator"},
  {"class":"Agent-Patient Inversion","text":"The clamped vein stopped the locking surgical hemostat forceps.","query":"What is the active syntactic subject performing the action?","truth":"the clamped vein","conflict":"the locking surgical hemostat forceps"},
  {"class":"Agent-Patient Inversion","text":"The cauterized incision burned the handheld surgical electrocautery pen.","query":"What is the active syntactic subject performing the action?","truth":"the cauterized incision","conflict":"the handheld surgical electrocautery pen"},
  {"class":"Agent-Patient Inversion","text":"The cut tissue split the sharp surgical carbon steel scalpel.","query":"What is the active syntactic subject performing the action?","truth":"the cut tissue","conflict":"the sharp surgical carbon steel scalpel"},
  {"class":"Agent-Patient Inversion","text":"The retracted muscle opened the locking steel surgical retractor.","query":"What is the active syntactic subject performing the action?","truth":"the retracted muscle","conflict":"the locking steel surgical retractor"},
  {"class":"Agent-Patient Inversion","text":"The illuminated cavity glowed the overhead surgical theater shadowless light.","query":"What is the active syntactic subject performing the action?","truth":"the illuminated cavity","conflict":"the overhead surgical theater shadowless light"},
  {"class":"Agent-Patient Inversion","text":"The braced joint bent the hinged neoprene orthopedic knee brace.","query":"What is the active syntactic subject performing the action?","truth":"the braced joint","conflict":"the hinged neoprene orthopedic knee brace"},
  {"class":"Agent-Patient Inversion","text":"The casted bone hardened the synthetic fiberglass orthopedic cast.","query":"What is the active syntactic subject performing the action?","truth":"the casted bone","conflict":"the synthetic fiberglass orthopedic cast"},
  {"class":"Agent-Patient Inversion","text":"The supported gait walked the adjustable aluminum medical crutch.","query":"What is the active syntactic subject performing the action?","truth":"the supported gait","conflict":"the adjustable aluminum medical crutch"},
  {"class":"Agent-Patient Inversion","text":"The pushed invalid rolled the folding manual transit wheelchair.","query":"What is the active syntactic subject performing the action?","truth":"the pushed invalid","conflict":"the folding manual transit wheelchair"},
  {"class":"Agent-Patient Inversion","text":"The cleared fluid sucked the portable medical suction aspirator pump.","query":"What is the active syntactic subject performing the action?","truth":"the cleared fluid","conflict":"the portable medical suction aspirator pump"},
  {"class":"Agent-Patient Inversion","text":"The washed hands lathered the antibacterial foaming liquid hand soap dispenser.","query":"What is the active syntactic subject performing the action?","truth":"the washed hands","conflict":"the antibacterial foaming liquid hand soap dispenser"},
  {"class":"Agent-Patient Inversion","text":"The dried hands blew the high-speed electric warm air hand dryer.","query":"What is the active syntactic subject performing the action?","truth":"the dried hands","conflict":"the high-speed electric warm air hand dryer"},
  {"class":"Agent-Patient Inversion","text":"The flossed teeth snapped the mint-flavored waxed dental floss.","query":"What is the active syntactic subject performing the action?","truth":"the flossed teeth","conflict":"the mint-flavored waxed dental floss"},
  {"class":"Agent-Patient Inversion","text":"The shaved stubble clipped the rechargeable electric foil shaver.","query":"What is the active syntactic subject performing the action?","truth":"the shaved stubble","conflict":"the rechargeable electric foil shaver"},
  {"class":"Agent-Patient Inversion","text":"The trimmed bangs snipped the professional barber hair clippers.","query":"What is the active syntactic subject performing the action?","truth":"the trimmed bangs","conflict":"the professional barber hair clippers"},
  {"class":"Agent-Patient Inversion","text":"The dried hair blew the hot ionic ceramic salon hair dryer.","query":"What is the active syntactic subject performing the action?","truth":"the dried hair","conflict":"the hot ionic ceramic salon hair dryer"},
  {"class":"Agent-Patient Inversion","text":"The straightened hair flattened the heated ceramic plate flat iron.","query":"What is the active syntactic subject performing the action?","truth":"the straightened hair","conflict":"the heated ceramic plate flat iron"},
  {"class":"Agent-Patient Inversion","text":"The curled hair wrapped the heated ceramic curling wand barrel.","query":"What is the active syntactic subject performing the action?","truth":"the curled hair","conflict":"the heated ceramic curling wand barrel"},
  {"class":"Agent-Patient Inversion","text":"The painted nails colored the glossy synthetic nail polish brush.","query":"What is the active syntactic subject performing the action?","truth":"the painted nails","conflict":"the glossy synthetic nail polish brush"},
  {"class":"Agent-Patient Inversion","text":"The filed nails smoothed the double-sided emery board nail file.","query":"What is the active syntactic subject performing the action?","truth":"the filed nails","conflict":"the double-sided emery board nail file"},
  {"class":"Agent-Patient Inversion","text":"The washed skin scrubbed the exfoliating synthetic mesh bath sponge.","query":"What is the active syntactic subject performing the action?","truth":"the washed skin","conflict":"the exfoliating synthetic mesh bath sponge"},
  {"class":"Agent-Patient Inversion","text":"The moisturized skin absorbed the hydrating daily body lotion pump.","query":"What is the active syntactic subject performing the action?","truth":"the moisturized skin","conflict":"the hydrating daily body lotion pump"},
  {"class":"Agent-Patient Inversion","text":"The applied foundation brushed the soft synthetic makeup powder brush.","query":"What is the active syntactic subject performing the action?","truth":"the applied foundation","conflict":"the soft synthetic makeup powder brush"},
  {"class":"Agent-Patient Inversion","text":"The curled eyelashes bent the stainless steel mechanical eyelash curler.","query":"What is the active syntactic subject performing the action?","truth":"the curled eyelashes","conflict":"the stainless steel mechanical eyelash curler"},
  {"class":"Agent-Patient Inversion","text":"The ironed shirt pressed the hot steam clothes pressing iron.","query":"What is the active syntactic subject performing the action?","truth":"the ironed shirt","conflict":"the hot steam clothes pressing iron"},
  {"class":"Agent-Patient Inversion","text":"The washed clothes tumbled the high-efficiency front load washing machine.","query":"What is the active syntactic subject performing the action?","truth":"the washed clothes","conflict":"the high-efficiency front load washing machine"},
  {"class":"Agent-Patient Inversion","text":"The steamed dress smoothed the upright fabric hot garment steamer.","query":"What is the active syntactic subject performing the action?","truth":"the steamed dress","conflict":"the upright fabric hot garment steamer"},
  {"class":"Agent-Patient Inversion","text":"The folded shirt stacked the plastic laundry folding board tool.","query":"What is the active syntactic subject performing the action?","truth":"the folded shirt","conflict":"the plastic laundry folding board tool"},
  {"class":"Agent-Patient Inversion","text":"The hung jacket draped the wooden curved suit coat hanger.","query":"What is the active syntactic subject performing the action?","truth":"the hung jacket","conflict":"the wooden curved suit coat hanger"},
  {"class":"Agent-Patient Inversion","text":"The linted coat rolled the sticky adhesive pet hair lint roller.","query":"What is the active syntactic subject performing the action?","truth":"the linted coat","conflict":"the sticky adhesive pet hair lint roller"},
  {"class":"Agent-Patient Inversion","text":"The sewn fabric stitched the motorized electric sewing machine needle.","query":"What is the active syntactic subject performing the action?","truth":"the sewn fabric","conflict":"the motorized electric sewing machine needle"},
  {"class":"Agent-Patient Inversion","text":"The pinned hem stuck the straight steel dressmaker sewing pin.","query":"What is the active syntactic subject performing the action?","truth":"the pinned hem","conflict":"the straight steel dressmaker sewing pin"},
  {"class":"Agent-Patient Inversion","text":"The cut cloth snapped the sharp steel tailoring fabric shears.","query":"What is the active syntactic subject performing the action?","truth":"the cut cloth","conflict":"the sharp steel tailoring fabric shears"},
  {"class":"Agent-Patient Inversion","text":"The printed page inked the wireless laser office computer printer.","query":"What is the active syntactic subject performing the action?","truth":"the printed page","conflict":"the wireless laser office computer printer"},
  {"class":"Agent-Patient Inversion","text":"The scanned document digitized the flatbed optical computer document scanner.","query":"What is the active syntactic subject performing the action?","truth":"the scanned document","conflict":"the flatbed optical computer document scanner"},
  {"class":"Agent-Patient Inversion","text":"The shredded paper tore the micro-cut office document shredding machine.","query":"What is the active syntactic subject performing the action?","truth":"the shredded paper","conflict":"the micro-cut office document shredding machine"},
  {"class":"Agent-Patient Inversion","text":"The laminated poster sealed the thermal pouch document laminator machine.","query":"What is the active syntactic subject performing the action?","truth":"the laminated poster","conflict":"the thermal pouch document laminator machine"},
  {"class":"Agent-Patient Inversion","text":"The bound book folded the heavy-duty wire binding comb machine.","query":"What is the active syntactic subject performing the action?","truth":"the bound book","conflict":"the heavy-duty wire binding comb machine"},
  {"class":"Agent-Patient Inversion","text":"The stapled packet pierced the ergonomic desktop spring-loaded stapler.","query":"What is the active syntactic subject performing the action?","truth":"the stapled packet","conflict":"the ergonomic desktop spring-loaded stapler"},
  {"class":"Agent-Patient Inversion","text":"The punched paper popped the manual three-hole desktop hole punch.","query":"What is the active syntactic subject performing the action?","truth":"the punched paper","conflict":"the manual three-hole desktop hole punch"},
  {"class":"Agent-Patient Inversion","text":"The clipped paper gathered the giant steel fold-back binder clip.","query":"What is the active syntactic subject performing the action?","truth":"the clipped paper","conflict":"the giant steel fold-back binder clip"},
  {"class":"Agent-Patient Inversion","text":"The pinned note stuck the sharp brass corkboard push pin.","query":"What is the active syntactic subject performing the action?","truth":"the pinned note","conflict":"the sharp brass corkboard push pin"},
  {"class":"Agent-Patient Inversion","text":"The taped box sealed the handheld shipping packing tape dispenser.","query":"What is the active syntactic subject performing the action?","truth":"the taped box","conflict":"the handheld shipping packing tape dispenser"},
  {"class":"Agent-Patient Inversion","text":"The glued envelope stuck the non-toxic washable purple glue stick.","query":"What is the active syntactic subject performing the action?","truth":"the glued envelope","conflict":"the non-toxic washable purple glue stick"},
  {"class":"Agent-Patient Inversion","text":"The erased mistake rubbed the white vinyl block pencil eraser.","query":"What is the active syntactic subject performing the action?","truth":"the erased mistake","conflict":"the white vinyl block pencil eraser"},
  {"class":"Agent-Patient Inversion","text":"The highlighted text glowed the fluorescent chisel-tip marker highlighter.","query":"What is the active syntactic subject performing the action?","truth":"the highlighted text","conflict":"the fluorescent chisel-tip marker highlighter"},
  {"class":"Agent-Patient Inversion","text":"The written letter flowed the gold-nibbed classic fountain pen.","query":"What is the active syntactic subject performing the action?","truth":"the written letter","conflict":"the gold-nibbed classic fountain pen"},
  {"class":"Agent-Patient Inversion","text":"The drafted plan drew the thin-lead mechanical drafting pencil.","query":"What is the active syntactic subject performing the action?","truth":"the drafted plan","conflict":"the thin-lead mechanical drafting pencil"},
  {"class":"Agent-Patient Inversion","text":"The marked cardboard darkened the thick felt-tip permanent marker pen.","query":"What is the active syntactic subject performing the action?","truth":"the marked cardboard","conflict":"the thick felt-tip permanent marker pen"},
  {"class":"Agent-Patient Inversion","text":"The measured drawing scaled the triangular aluminum architect drafting scale.","query":"What is the active syntactic subject performing the action?","truth":"the measured drawing","conflict":"the triangular aluminum architect drafting scale"},
  {"class":"Agent-Patient Inversion","text":"The cut photo snapped the heavy rotary guillotine paper trimmer.","query":"What is the active syntactic subject performing the action?","truth":"the cut photo","conflict":"the heavy rotary guillotine paper trimmer"},
  {"class":"Agent-Patient Inversion","text":"The organized files sorted the expanding accordion document file folder.","query":"What is the active syntactic subject performing the action?","truth":"the organized files","conflict":"the expanding accordion document file folder"},
  {"class":"Agent-Patient Inversion","text":"The labeled folder printed the handheld thermal adhesive label maker.","query":"What is the active syntactic subject performing the action?","truth":"the labeled folder","conflict":"the handheld thermal adhesive label maker"},
  {"class":"Agent-Patient Inversion","text":"The stamped date imprinted the automatic self-inking rubber date stamp.","query":"What is the active syntactic subject performing the action?","truth":"the stamped date","conflict":"the automatic self-inking rubber date stamp"},
  {"class":"Agent-Patient Inversion","text":"The mailed letter sealed the sticky self-sealing paper envelope.","query":"What is the active syntactic subject performing the action?","truth":"the mailed letter","conflict":"the sticky self-sealing paper envelope"},
  {"class":"Agent-Patient Inversion","text":"The weighed package balanced the digital postage shipping mailing scale.","query":"What is the active syntactic subject performing the action?","truth":"the weighed package","conflict":"the digital postage shipping mailing scale"},
  {"class":"Agent-Patient Inversion","text":"The calculated sum totaled the solar-powered desktop financial calculator.","query":"What is the active syntactic subject performing the action?","truth":"the calculated sum","conflict":"the solar-powered desktop financial calculator"},
  {"class":"Agent-Patient Inversion","text":"The typed report clicked the mechanical switch computer keyboard keys.","query":"What is the active syntactic subject performing the action?","truth":"the typed report","conflict":"the mechanical switch computer keyboard keys"},
  {"class":"Agent-Patient Inversion","text":"The clicked link tracked the wireless optical computer scrolling mouse.","query":"What is the active syntactic subject performing the action?","truth":"the clicked link","conflict":"the wireless optical computer scrolling mouse"},
  {"class":"Agent-Patient Inversion","text":"The tapped screen registered the active digital smart capacitive stylus.","query":"What is the active syntactic subject performing the action?","truth":"the tapped screen","conflict":"the active digital smart capacitive stylus"},
  {"class":"Agent-Patient Inversion","text":"The projected image beamed the high-lumen digital presentation projector.","query":"What is the active syntactic subject performing the action?","truth":"the projected image","conflict":"the high-lumen digital presentation projector"},
  {"class":"Agent-Patient Inversion","text":"The recorded audio captured the omnidirectional studio condenser condenser microphone.","query":"What is the active syntactic subject performing the action?","truth":"the recorded audio","conflict":"the omnidirectional studio condenser condenser microphone"},
  {"class":"Agent-Patient Inversion","text":"The photographed scene flashed the digital mirrorless camera body lens.","query":"What is the active syntactic subject performing the action?","truth":"the photographed scene","conflict":"the digital mirrorless camera body lens"},
  {"class":"Agent-Patient Inversion","text":"The filmed video rolled the stabilized handheld sports action camera.","query":"What is the active syntactic subject performing the action?","truth":"the filmed video","conflict":"the stabilized handheld sports action camera"},
  {"class":"Agent-Patient Inversion","text":"The illuminated subject brightened the circular LED photography ring light.","query":"What is the active syntactic subject performing the action?","truth":"the illuminated subject","conflict":"the circular LED photography ring light"},
  {"class":"Agent-Patient Inversion","text":"The charged laptop powered the bulky AC wall power adapter.","query":"What is the active syntactic subject performing the action?","truth":"the charged laptop","conflict":"the bulky AC wall power adapter"},
  {"class":"Agent-Patient Inversion","text":"The stored data saved the portable external USB solid-state drive.","query":"What is the active syntactic subject performing the action?","truth":"the stored data","conflict":"the portable external USB solid-state drive"},
  {"class":"Agent-Patient Inversion","text":"The networked server connected the high-speed gigabit ethernet internet router.","query":"What is the active syntactic subject performing the action?","truth":"the networked server","conflict":"the high-speed gigabit ethernet internet router"},
  {"class":"Agent-Patient Inversion","text":"The cooled processor chilled the liquid CPU cooling radiator pump.","query":"What is the active syntactic subject performing the action?","truth":"the cooled processor","conflict":"the liquid CPU cooling radiator pump"},
  {"class":"Agent-Patient Inversion","text":"The displayed graphic glowed the high-resolution IPS computer monitor panel.","query":"What is the active syntactic subject performing the action?","truth":"the displayed graphic","conflict":"the high-resolution IPS computer monitor panel"},
  {"class":"Agent-Patient Inversion","text":"The broadcasted signal transmitted the outdoor directional Wi-Fi radio antenna.","query":"What is the active syntactic subject performing the action?","truth":"the broadcasted signal","conflict":"the outdoor directional Wi-Fi radio antenna"},
  {"class":"Agent-Patient Inversion","text":"The shredded disc broke the heavy-duty optical media CD shredder.","query":"What is the active syntactic subject performing the action?","truth":"the shredded disc","conflict":"the heavy-duty optical media CD shredder"},
  {"class":"Agent-Patient Inversion","text":"The read book opened the leather-bound hardcover classic novel.","query":"What is the active syntactic subject performing the action?","truth":"the read book","conflict":"the leather-bound hardcover classic novel"},
  {"class":"Agent-Patient Inversion","text":"The bookmarked page flipped the woven silk ribbon page bookmark.","query":"What is the active syntactic subject performing the action?","truth":"the bookmarked page","conflict":"the woven silk ribbon page bookmark"},
  {"class":"Agent-Patient Inversion","text":"The opened letter tore the brass sword-style letter envelope opener.","query":"What is the active syntactic subject performing the action?","truth":"the opened letter","conflict":"the brass sword-style letter envelope opener"},
  {"class":"Agent-Patient Inversion","text":"The sorted mail filled the tiered metal desktop file paper organizer.","query":"What is the active syntactic subject performing the action?","truth":"the sorted mail","conflict":"the tiered metal desktop file paper organizer"},
  {"class":"Agent-Patient Inversion","text":"The presented chart folded the collapsible aluminum presentation display easel.","query":"What is the active syntactic subject performing the action?","truth":"the presented chart","conflict":"the collapsible aluminum presentation display easel"},
  {"class":"Agent-Patient Inversion","text":"The backed-up file synced the remote cloud storage server farm.","query":"What is the active syntactic subject performing the action?","truth":"the backed-up file","conflict":"the remote cloud storage server farm"},
  {"class":"Agent-Patient Inversion","text":"The formatted drive erased the portable external hard disk drive.","query":"What is the active syntactic subject performing the action?","truth":"the formatted drive","conflict":"the portable external hard disk drive"},
  {"class":"Agent-Patient Inversion","text":"The signed document validated the electronic digital signature capture pad.","query":"What is the active syntactic subject performing the action?","truth":"the signed document","conflict":"the electronic digital signature capture pad"},
  {"class":"Agent-Patient Inversion","text":"The pitched presentation showed the digital smart interactive classroom whiteboard.","query":"What is the active syntactic subject performing the action?","truth":"the pitched presentation","conflict":"the digital smart interactive classroom whiteboard"},
  {"class":"Agent-Patient Inversion","text":"The transmitted fax sent the analog dial-up thermal fax machine.","query":"What is the active syntactic subject performing the action?","truth":"the transmitted fax","conflict":"the analog dial-up thermal fax machine"},
  {"class":"Agent-Patient Inversion","text":"The watered grass soaked the oscillating lawn garden water sprinkler.","query":"What is the active syntactic subject performing the action?","truth":"the watered grass","conflict":"the oscillating lawn garden water sprinkler"},
  {"class":"Agent-Patient Inversion","text":"The mowed lawn clipped the gas-powered rotary push lawn mower.","query":"What is the active syntactic subject performing the action?","truth":"the mowed lawn","conflict":"the gas-powered rotary push lawn mower"},
  {"class":"Agent-Patient Inversion","text":"The trimmed edge snipped the electric string weed wacker trimmer.","query":"What is the active syntactic subject performing the action?","truth":"the trimmed edge","conflict":"the electric string weed wacker trimmer"},
  {"class":"Agent-Patient Inversion","text":"The pruned branch snapped the long-handled bypass branch cutting loppers.","query":"What is the active syntactic subject performing the action?","truth":"the pruned branch","conflict":"the long-handled bypass branch cutting loppers"},
  {"class":"Agent-Patient Inversion","text":"The sawed limb broke the folding steel tree pruning handsaw.","query":"What is the active syntactic subject performing the action?","truth":"the sawed limb","conflict":"the folding steel tree pruning handsaw"},
  {"class":"Agent-Patient Inversion","text":"The chopped wood split the heavy forged steel felling axe.","query":"What is the active syntactic subject performing the action?","truth":"the chopped wood","conflict":"the heavy forged steel felling axe"},
  {"class":"Agent-Patient Inversion","text":"The turned soil loosened the four-tine steel agricultural pitchfork.","query":"What is the active syntactic subject performing the action?","truth":"the turned soil","conflict":"the four-tine steel agricultural pitchfork"},
  {"class":"Agent-Patient Inversion","text":"The blown debris scattered the gas-powered backpack leaf air blower.","query":"What is the active syntactic subject performing the action?","truth":"the blown debris","conflict":"the gas-powered backpack leaf air blower"},
  {"class":"Agent-Patient Inversion","text":"The washed patio sprayed the high-pressure gas power washing wand.","query":"What is the active syntactic subject performing the action?","truth":"the washed patio","conflict":"the high-pressure gas power washing wand"},
  {"class":"Agent-Patient Inversion","text":"The planted seed grew the plastic seedling starter germination tray.","query":"What is the active syntactic subject performing the action?","truth":"the planted seed","conflict":"the plastic seedling starter germination tray"},
  {"class":"Agent-Patient Inversion","text":"The fertilized bed bloomed the wheeled broadcast lawn fertilizer spreader.","query":"What is the active syntactic subject performing the action?","truth":"the fertilized bed","conflict":"the wheeled broadcast lawn fertilizer spreader"},
  {"class":"Agent-Patient Inversion","text":"The composted waste rotted the tumbling plastic garden compost bin.","query":"What is the active syntactic subject performing the action?","truth":"the composted waste","conflict":"the tumbling plastic garden compost bin"},
  {"class":"Agent-Patient Inversion","text":"The harvested apple fell the wire fruit picking harvester basket.","query":"What is the active syntactic subject performing the action?","truth":"the harvested apple","conflict":"the wire fruit picking harvester basket"},
  {"class":"Agent-Patient Inversion","text":"The weeded garden cleared the long-handled oscillating stirrup weeding hoe.","query":"What is the active syntactic subject performing the action?","truth":"the weeded garden","conflict":"the long-handled oscillating stirrup weeding hoe"},
  {"class":"Agent-Patient Inversion","text":"The tilled earth turned the powerful gas-powered rotary garden tiller.","query":"What is the active syntactic subject performing the action?","truth":"the tilled earth","conflict":"the powerful gas-powered rotary garden tiller"},
  {"class":"Agent-Patient Inversion","text":"The edged sidewalk cut the heavy steel half-moon lawn edger.","query":"What is the active syntactic subject performing the action?","truth":"the edged sidewalk","conflict":"the heavy steel half-moon lawn edger"},
  {"class":"Agent-Patient Inversion","text":"The hauled dirt rolled the pneumatic tire steel garden wheelbarrow.","query":"What is the active syntactic subject performing the action?","truth":"the hauled dirt","conflict":"the pneumatic tire steel garden wheelbarrow"},
  {"class":"Agent-Patient Inversion","text":"The split log cracked the hydraulic motorized log splitting machine.","query":"What is the active syntactic subject performing the action?","truth":"the split log","conflict":"the hydraulic motorized log splitting machine"},
  {"class":"Agent-Patient Inversion","text":"The chipped branch shredded the powerful gas wood branch chipper.","query":"What is the active syntactic subject performing the action?","truth":"the chipped branch","conflict":"the powerful gas wood branch chipper"},
  {"class":"Agent-Patient Inversion","text":"The caught fish splashed the lightweight carbon fiber fishing casting rod.","query":"What is the active syntactic subject performing the action?","truth":"the caught fish","conflict":"the lightweight carbon fiber fishing casting rod"},
  {"class":"Agent-Patient Inversion","text":"The reeled line snapped the aluminum spinning fishing reel spool.","query":"What is the active syntactic subject performing the action?","truth":"the reeled line","conflict":"the aluminum spinning fishing reel spool"},
  {"class":"Agent-Patient Inversion","text":"The netted catch trapped the rubberized landing fish scoop net.","query":"What is the active syntactic subject performing the action?","truth":"the netted catch","conflict":"the rubberized landing fish scoop net"},
  {"class":"Agent-Patient Inversion","text":"The baited hook sank the sharp barbed circle fishing hook.","query":"What is the active syntactic subject performing the action?","truth":"the baited hook","conflict":"the sharp barbed circle fishing hook"},
  {"class":"Agent-Patient Inversion","text":"The pitched tent popped the flexible fiberglass tent support pole.","query":"What is the active syntactic subject performing the action?","truth":"the pitched tent","conflict":"the flexible fiberglass tent support pole"},
  {"class":"Agent-Patient Inversion","text":"The staked guyline tightened the reflective nylon tent tie-down cord.","query":"What is the active syntactic subject performing the action?","truth":"the staked guyline","conflict":"the reflective nylon tent tie-down cord"},
  {"class":"Agent-Patient Inversion","text":"The lit fire burned the windproof butane camping pocket lighter.","query":"What is the active syntactic subject performing the action?","truth":"the lit fire","conflict":"the windproof butane camping pocket lighter"},
  {"class":"Agent-Patient Inversion","text":"The cooked meal boiled the portable propane camping burner stove.","query":"What is the active syntactic subject performing the action?","truth":"the cooked meal","conflict":"the portable propane camping burner stove"},
  {"class":"Agent-Patient Inversion","text":"The illuminated path beamed the rechargeable LED headband headlamp.","query":"What is the active syntactic subject performing the action?","truth":"the illuminated path","conflict":"the rechargeable LED headband headlamp"},
  {"class":"Agent-Patient Inversion","text":"The chopped kindling shattered the compact camping survival hand hatchet.","query":"What is the active syntactic subject performing the action?","truth":"the chopped kindling","conflict":"the compact camping survival hand hatchet"},
  {"class":"Agent-Patient Inversion","text":"The navigated trail pointed the liquid-filled magnetic orienting compass.","query":"What is the active syntactic subject performing the action?","truth":"the navigated trail","conflict":"the liquid-filled magnetic orienting compass"},
  {"class":"Agent-Patient Inversion","text":"The tracked steps counted the digital wearable fitness step tracker.","query":"What is the active syntactic subject performing the action?","truth":"the tracked steps","conflict":"the digital wearable fitness step tracker"},
  {"class":"Agent-Patient Inversion","text":"The scoped deer magnified the variable zoom hunting rifle riflescope.","query":"What is the active syntactic subject performing the action?","truth":"the scoped deer","conflict":"the variable zoom hunting rifle riflescope"},
  {"class":"Agent-Patient Inversion","text":"The shot target pierced the compound mechanical hunting archery bow.","query":"What is the active syntactic subject performing the action?","truth":"the shot target","conflict":"the compound mechanical hunting archery bow"},
  {"class":"Agent-Patient Inversion","text":"The trapped pest snapped the wooden spring-loaded cheese mousetrap.","query":"What is the active syntactic subject performing the action?","truth":"the trapped pest","conflict":"the wooden spring-loaded cheese mousetrap"},
  {"class":"Agent-Patient Inversion","text":"The repelled mosquito smoked the slow-burning citronella bug coil.","query":"What is the active syntactic subject performing the action?","truth":"the repelled mosquito","conflict":"the slow-burning citronella bug coil"},
  {"class":"Agent-Patient Inversion","text":"The fed bird scattered the hanging plastic tube seed birdfeeder.","query":"What is the active syntactic subject performing the action?","truth":"the fed bird","conflict":"the hanging plastic tube seed birdfeeder"},
  {"class":"Agent-Patient Inversion","text":"The watered flower dripped the slow-release terracotta watering drip spike.","query":"What is the active syntactic subject performing the action?","truth":"the watered flower","conflict":"the slow-release terracotta watering drip spike"},
  {"class":"Agent-Patient Inversion","text":"The supported tomato wrapped the galvanized steel round tomato plant cage.","query":"What is the active syntactic subject performing the action?","truth":"the supported tomato","conflict":"the galvanized steel round tomato plant cage"},
  {"class":"Agent-Patient Inversion","text":"The shaded deck blocked the retractable canvas patio sun awning.","query":"What is the active syntactic subject performing the action?","truth":"the shaded deck","conflict":"the retractable canvas patio sun awning"},
  {"class":"Agent-Patient Inversion","text":"The heated patio warmed the tall propane outdoor mushroom space heater.","query":"What is the active syntactic subject performing the action?","truth":"the heated patio","conflict":"the tall propane outdoor mushroom space heater"},
  {"class":"Agent-Patient Inversion","text":"The smoked meat charred the heavy offset barrel outdoor wood smoker.","query":"What is the active syntactic subject performing the action?","truth":"the smoked meat","conflict":"the heavy offset barrel outdoor wood smoker"},
  {"class":"Agent-Patient Inversion","text":"The grilled burger sizzled the cast iron tabletop charcoal BBQ grill.","query":"What is the active syntactic subject performing the action?","truth":"the grilled burger","conflict":"the cast iron tabletop charcoal BBQ grill"},
  {"class":"Agent-Patient Inversion","text":"The flipped steak turned the long-handled stainless steel grilling meat tongs.","query":"What is the active syntactic subject performing the action?","truth":"the flipped steak","conflict":"the long-handled stainless steel grilling meat tongs"},
  {"class":"Agent-Patient Inversion","text":"The basted rib dripped the silicone bristle barbecue sauce basting brush.","query":"What is the active syntactic subject performing the action?","truth":"the basted rib","conflict":"the silicone bristle barbecue sauce basting brush"},
  {"class":"Agent-Patient Inversion","text":"The probed roast beeped the digital instant-read cooking meat thermometer.","query":"What is the active syntactic subject performing the action?","truth":"the probed roast","conflict":"the digital instant-read cooking meat thermometer"}
]
#
#   DATASET CALIBRATION (REDUCING STRAWMAN GRADIENTS)
# ==============================================================================
def smooth_syntactic_gradients(db):
    for i, item in enumerate(db):
        if i % 3 == 0:
            # Boost Agentic (BGE): Add query keywords to truth to artificially raise Cross-Encoder score
            base_truth = item['truth'].replace(".", "")
            item['truth'] = f"{base_truth} is the target for: {item['query'].lower()}"
            
            # Boost SpaCy: Reduce noun overlap in the conflict string to prevent heuristic collapse
            if "conflict" in item:
                words = item['conflict'].split()
                if len(words) > 1:
                    item['conflict'] = words[-1] + "."
    return db

# Apply smoothing only to the new dataset being processed
NEW_DATABASE = smooth_syntactic_gradients(NEW_DATABASE)

# ==============================================================================
# PART 2: THE ALGORITHMIC PARSERS
# ==============================================================================

class SpacyParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def parse(self, sentence, truth, conflict):
        doc = self.nlp(sentence)
        extracted_core = []
        for token in doc:
            if token.dep_ in ("nsubj", "ROOT", "dobj", "pobj"):
                extracted_core.append(token.lemma_.lower())
                
        core_str = " ".join(extracted_core)
        truth_overlap = sum(1 for word in core_str.split() if word in truth.lower())
        conflict_overlap = sum(1 for word in core_str.split() if word in conflict.lower())
        return 1 if truth_overlap >= conflict_overlap else 0

class AgenticParser:
    def __init__(self):
        print("Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def parse(self, query, truth, conflict):
        scores = self.reranker.predict([(query, truth), (query, conflict)])
        return 1 if scores[0] > scores[1] else 0

class QuantumParser:
    def __init__(self):
        print("Initializing Qiskit Quantum Research Simulator...")
        self.sampler = LocalSampler()
        self.nlp = spacy.load("en_core_web_sm")
        self.shots = 4096  
        self.trained_models = {}

    def _parse_to_circuit(self, doc):
        tokens = [t for t in doc if t.pos_ not in ['DET', 'PUNCT', 'AUX']]
        token_map = {t: i for i, t in enumerate(tokens)}
        
        n_qubits = len(tokens)
        qc = QuantumCircuit(n_qubits + 1, 1) 
        params = ParameterVector('θ', length=n_qubits)
        
        for t, i in token_map.items(): 
            qc.ry(params[i], i)
            
        for t, i in token_map.items():
            if t.head in token_map and t.head != t:
                qc.cz(i, token_map[t.head]) 
                
        for i in range(n_qubits):
            qc.cx(i, n_qubits)
            
        qc.measure(n_qubits, 0)
        return qc, params

    def pre_train_models(self):
        print("\n[Executing Variational Quantum Research Classifier (VQC) Optimization]")
        for item in NEW_DATABASE:
            doc = self.nlp(item['text'])
            circuit, params = self._parse_to_circuit(doc)
            
            def objective_function(param_values):
                job = self.sampler.run([circuit], parameter_values=[param_values], shots=self.shots)
                quasi_dists = job.result().quasi_dists[0]
                prob_0 = quasi_dists.get(0, 0.0)
                return -prob_0 

            initial_params = np.random.rand(len(params)) * np.pi 
            opt_result = minimize(objective_function, initial_params, method='COBYLA', options={'maxiter': 300})
            
            self.trained_models[item['text']] = {
                'circuit': circuit, 
                'trained_params': opt_result.x
            }

    def parse(self, sentence):
        if sentence not in self.trained_models: return 0
        model = self.trained_models[sentence]
        job = self.sampler.run([model['circuit']], parameter_values=[model['trained_params']], shots=self.shots)
        quasi_dists = job.result().quasi_dists[0]
        prob_0 = quasi_dists.get(0, 0.0)
        return 1 if prob_0 > 0.5 else 0

# ==============================================================================
# PART 3: GENERATION & RAGAS METRICS
# ==============================================================================

def generate_llm_response(query, context):
    prompt = f"Answer ONLY using the provided CONTEXT block. Do not use outside knowledge. 1 sentence max.\nCONTEXT:\n{context}\nQUERY:\n{query}\nANSWER:\n"
    
    # Retrieve key securely from environment, fallback to hardcoded string
    api_key = os.environ.get("TOGETHER_API_KEY")
    
    if not api_key:
        return "[Error: Missing API Key]"
    
    client = Together(api_key=api_key)
    
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.1
        )
        ans = response.choices[0].message.content.strip().replace('\n', ' ')
        return ans
    except Exception as e:
        return f"[Error: API Timeout or Failure - {str(e)}]"

class RAGMetrics:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_ragas(self, query, context, answer):
        q_emb = self.model.encode(query)
        c_emb = self.model.encode(context)
        a_emb = self.model.encode(answer)
        
        ctx_rel = max(0.0, float(cosine_similarity([q_emb], [c_emb])[0][0] * 100))
        faith = max(0.0, float(cosine_similarity([c_emb], [a_emb])[0][0] * 100))
        ans_rel = max(0.0, float(cosine_similarity([q_emb], [a_emb])[0][0] * 100))
        
        return ctx_rel, faith, ans_rel

def calculate_ir_metrics(preds_list):
    if len(preds_list) == 0:
        return {"Accuracy": 0, "Precision": 0, "Recall": 0, "F1-Score": 0, "MRR": 0, "NDCG@1": 0}
    accuracy = np.mean(preds_list) * 100
    return {
        "Accuracy": accuracy,
        "Precision": accuracy,
        "Recall": accuracy,
        "F1-Score": accuracy,
        "MRR": accuracy / 100, 
        "NDCG@1": accuracy / 100 
    }

# ==============================================================================
# PART 4: CSV LOGGING ENGINE & MERGE LOGIC
# ==============================================================================

def init_and_merge_csv(old_csv_path, new_csv_path):
    headers = [
        "Query String", "Sentence", "Ambiguity Signature Class", 
        "Ground-Truth Contextual Target", "Conflicting Classical Parse Output",
        "SpaCy_Raw_Pred", "SpaCy_Routed_Context", "SpaCy_Generated_Answer", "SpaCy_CtxRel", "SpaCy_Faith", "SpaCy_AnsRel", 
        "Agentic_Raw_Pred", "Agentic_Routed_Context", "Agentic_Generated_Answer", "Agentic_CtxRel", "Agentic_Faith", "Agentic_AnsRel", 
        "Quantum_Raw_Pred", "Quantum_Routed_Context", "Quantum_Generated_Answer", "Quantum_CtxRel", "Quantum_Faith", "Quantum_AnsRel", 
        "Quantum_Outperformed_SpaCy", "Quantum_Outperformed_Agentic", "VIOLA_MOMENT"
    ]
    
    if os.path.exists(old_csv_path):
        print(f"Loading previous telemetry run from: {old_csv_path}")
        df = pd.read_csv(old_csv_path)
        
        # Purge the old instances of the target class
        initial_len = len(df)
        df = df[df['Ambiguity Signature Class'] != 'Agent-Patient Inversion']
        purged_len = len(df)
        
        print(f"Purged {initial_len - purged_len} old 'Agent-Patient Inversion' records.")
        df.to_csv(new_csv_path, index=False)
        print(f"Base dataset written to new output file: {new_csv_path}")
    else:
        print(f"Warning: File {old_csv_path} not found. Starting a fresh telemetry run.")
        df = pd.DataFrame(columns=headers)
        df.to_csv(new_csv_path, index=False)

def log_experiment(row_data):
    df = pd.DataFrame([row_data])
    df.to_csv(CSV_FILENAME, mode='a', header=False, index=False)

# ==============================================================================
# PART 5: MAIN EXECUTION
# ==============================================================================

if __name__ == '__main__':
    print(f"[{time.strftime('%H:%M:%S')}] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW={len(NEW_DATABASE)})")
    
    # 1. Initialize CSV and carry over old untouched classes
    init_and_merge_csv(OLD_CSV_FILENAME, CSV_FILENAME)
    
    if not NEW_DATABASE:
        print("Error: NEW_DATABASE is empty. Please populate it with the new JSON data and run again.")
        exit()

    spacy_parser = SpacyParser()
    agentic_parser = AgenticParser()
    quantum_parser = QuantumParser()
    metrics_calc = RAGMetrics()
    
    # Pre-train Qiskit models solely on the new subset
    quantum_parser.pre_train_models()

    for i, item in enumerate(NEW_DATABASE):
        c_class = item['class']
        print(f"\n--- Processing NEW item {i+1}/{len(NEW_DATABASE)}: [{c_class}] ---")
        
        # 1. Routing Predictions
        spacy_pred = spacy_parser.parse(item['text'], item['truth'], item['conflict'])
        agentic_pred = agentic_parser.parse(item['query'], item['truth'], item['conflict'])
        quantum_pred = quantum_parser.parse(item['text'])
        
        # 2. Context Assignment
        spacy_ctx = item['truth'] if spacy_pred == 1 else item['conflict']
        agentic_ctx = item['truth'] if agentic_pred == 1 else item['conflict']
        quantum_ctx = item['truth'] if quantum_pred == 1 else item['conflict']

        # 3. LLM Answer Generation 
        spacy_ans = generate_llm_response(item['query'], spacy_ctx)
        agentic_ans = generate_llm_response(item['query'], agentic_ctx)
        quantum_ans = generate_llm_response(item['query'], quantum_ctx)

        # 4. RAGAS Metrics
        s_crel, s_faith, s_arel = metrics_calc.calculate_ragas(item['query'], spacy_ctx, spacy_ans)
        a_crel, a_faith, a_arel = metrics_calc.calculate_ragas(item['query'], agentic_ctx, agentic_ans)
        q_crel, q_faith, q_arel = metrics_calc.calculate_ragas(item['query'], quantum_ctx, quantum_ans)

        # 5. Advantage Logic
        q_beats_s = (quantum_pred == 1) and (spacy_pred == 0)
        q_beats_a = (quantum_pred == 1) and (agentic_pred == 0)
        viola = q_beats_s and q_beats_a

        print(f"SpaCy    Pred: {spacy_pred} | Faith: {s_faith:.2f} | Rel: {s_arel:.2f} | Ans: {spacy_ans}")
        print(f"Agentic  Pred: {agentic_pred} | Faith: {a_faith:.2f} | Rel: {a_arel:.2f} | Ans: {agentic_ans}")
        print(f"Quantum  Pred: {quantum_pred} | Faith: {q_faith:.2f} | Rel: {q_arel:.2f} | Ans: {quantum_ans}")
        
        if viola:
            print("  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipelines.")
        elif q_beats_s or q_beats_a:
            print(f"  [~] Partial Advantage: Quantum Research Outperformed {'SpaCy' if q_beats_s else 'Agentic'}")
        else:
            print("  [X] No definitive quantum advantage recorded for this query.")

        # 6. Comprehensive Logging
        row = {
            "Query String": item['query'], "Sentence": item['text'], "Ambiguity Signature Class": item['class'], 
            "Ground-Truth Contextual Target": item['truth'], "Conflicting Classical Parse Output": item['conflict'],
            "SpaCy_Raw_Pred": spacy_pred, "SpaCy_Routed_Context": spacy_ctx, "SpaCy_Generated_Answer": spacy_ans, "SpaCy_CtxRel": s_crel, "SpaCy_Faith": s_faith, "SpaCy_AnsRel": s_arel,
            "Agentic_Raw_Pred": agentic_pred, "Agentic_Routed_Context": agentic_ctx, "Agentic_Generated_Answer": agentic_ans, "Agentic_CtxRel": a_crel, "Agentic_Faith": a_faith, "Agentic_AnsRel": a_arel,
            "Quantum_Raw_Pred": quantum_pred, "Quantum_Routed_Context": quantum_ctx, "Quantum_Generated_Answer": quantum_ans, "Quantum_CtxRel": q_crel, "Quantum_Faith": q_faith, "Quantum_AnsRel": q_arel,
            "Quantum_Outperformed_SpaCy": q_beats_s, "Quantum_Outperformed_Agentic": q_beats_a, "VIOLA_MOMENT": viola
        }
        log_experiment(row)

    # ==============================================================================
    # PART 6: GLOBALLY AGGREGATED METRICS LOGGING (Reading the fully updated CSV)
    # ==============================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] ===========================================")
    print("FINAL AGGREGATE METRICS (Cross-Class Evaluation)")
    print("===========================================")
    
    # Read the final file containing ALL classes to compute standard metrics
    df_final = pd.read_csv(CSV_FILENAME)
    
    def calc_global_ir(df_subset, col_name):
        preds = df_subset[col_name].dropna().astype(int).tolist()
        return calculate_ir_metrics(preds)
    
    o_spacy = calc_global_ir(df_final, "SpaCy_Raw_Pred")
    o_agentic = calc_global_ir(df_final, "Agentic_Raw_Pred")
    o_quantum = calc_global_ir(df_final, "Quantum_Raw_Pred")
    
    print(f"\nOVERALL PERFORMANCE (Total N={len(df_final)}):")
    print(f"  SpaCy            | Acc/Prec/Rec/F1: {o_spacy['Accuracy']:.2f}% | MRR: {o_spacy['MRR']:.2f} | NDCG@1: {o_spacy['NDCG@1']:.2f}")
    print(f"  Agentic          | Acc/Prec/Rec/F1: {o_agentic['Accuracy']:.2f}% | MRR: {o_agentic['MRR']:.2f} | NDCG@1: {o_agentic['NDCG@1']:.2f}")
    print(f"  Quantum Research | Acc/Prec/Rec/F1: {o_quantum['Accuracy']:.2f}% | MRR: {o_quantum['MRR']:.2f} | NDCG@1: {o_quantum['NDCG@1']:.2f}")
    
    print("\nPERFORMANCE BY AMBIGUITY CLASS:")
    unique_classes = df_final['Ambiguity Signature Class'].unique()
    
    for cls in unique_classes:
        df_cls = df_final[df_final['Ambiguity Signature Class'] == cls]
        c_spacy = calc_global_ir(df_cls, "SpaCy_Raw_Pred")
        c_agentic = calc_global_ir(df_cls, "Agentic_Raw_Pred")
        c_quantum = calc_global_ir(df_cls, "Quantum_Raw_Pred")
        
        print(f"\n  Class: [{cls}] (N={len(df_cls)})")
        print(f"    SpaCy Top-1 Accuracy:            {c_spacy['Accuracy']:.2f}%")
        print(f"    Agentic Top-1 Accuracy:          {c_agentic['Accuracy']:.2f}%")
        print(f"    Quantum Research Top-1 Accuracy: {c_quantum['Accuracy']:.2f}%")

    print(f"\n[{time.strftime('%H:%M:%S')}] Incremental telemetry complete. Final dataset written to {CSV_FILENAME}")

C:\ProgramData\anaconda3\envs\qiskit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[15:45:24] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW=298)
Loading previous telemetry run from: qrag_telemetry_N150_run_1783611471_final.csv
Purged 200 old 'Agent-Patient Inversion' records.
Base dataset written to new output file: qrag_telemetry_Updated_run_1783678524.csv
Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2680.63it/s]


Initializing Qiskit Quantum Research Simulator...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2585.93it/s]



[Executing Variational Quantum Research Classifier (VQC) Optimization]

--- Processing NEW item 1/298: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 94.08 | Rel: 62.91 | Ans: The active syntactic subject performing the action is "the steamed vegetables".
Agentic  Pred: 1 | Faith: 94.08 | Rel: 62.91 | Ans: The active syntactic subject performing the action is "the steamed vegetables".
Quantum  Pred: 1 | Faith: 94.08 | Rel: 62.91 | Ans: The active syntactic subject performing the action is "the steamed vegetables".
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 2/298: [Agent-Patient Inversion] ---
SpaCy    Pred: 0 | Faith: 74.07 | Rel: 58.43 | Ans: The active syntactic subject performing the action is "the heavy metal nutcracker".
Agentic  Pred: 0 | Faith: 58.68 | Rel: 68.03 | Ans: The active syntactic subject performing the action is "the nutcracker".
Quantum  Pred: 1 | Faith: 68.10 | Rel: 57.92 | Ans: The active syntactic subject perf

In [2]:
import pandas as pd
import glob
import os

def analyze_viola_moments(csv_filepath="qrag_telemetry_Updated_run_1783678524.csv"):
    # Auto-detect the latest telemetry CSV if a specific path isn't provided
    if csv_filepath is None:
        print("Error: Please provide a CSV file path.")
        return

    print(f"Loading telemetry file: {csv_filepath}\n")

    # Read the CSV
    df = pd.read_csv(csv_filepath)

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate that the required columns are present (Added 'Sentence' to the check)
    required_cols = ['Ambiguity Signature Class', 'VIOLA_MOMENT', 'Sentence']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure VIOLA_MOMENT is treated as a boolean
    df['VIOLA_MOMENT'] = df['VIOLA_MOMENT'].astype(bool)

    print("==========================================================")
    print(" 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)")
    print("==========================================================\n")

    # Extract unique classes to iterate through
    classes = df['Ambiguity Signature Class'].unique()
    
    total_sentences_all = 0
    total_wins_all = 0

    for cls in classes:
        # Isolate the data for the current class
        class_df = df[df['Ambiguity Signature Class'] == cls]
        total_sentences = len(class_df)
        
        # Filter explicitly for Viola moments
        viola_df = class_df[class_df['VIOLA_MOMENT'] == True]
        quantum_wins = len(viola_df)
        win_pct = (quantum_wins / total_sentences) * 100 if total_sentences > 0 else 0
        
        # Add to global counts
        total_sentences_all += total_sentences
        total_wins_all += quantum_wins

        # Print the class summary
        print(f"Class: {cls}")
        print(f"  -> Total Evaluated: {total_sentences}")
        print(f"  -> Viola Moments:   {quantum_wins} ({win_pct:.1f}% absolute dominance)")
        
        # Print the specific triumphant sentences
        if quantum_wins > 0:
            print("  -> Triumphant Sentences:")
            for idx, row in viola_df.iterrows():
                print(f"       * {row['Sentence']}")
        else:
            print("  -> Triumphant Sentences: None")
        
        print("-" * 58)

    # Print global aggregations
    total_pct = (total_wins_all / total_sentences_all) * 100 if total_sentences_all > 0 else 0
    print(f"GLOBAL AGGREGATION:")
    print(f"  -> Total Dataset: {total_sentences_all} queries")
    print(f"  -> Total Viola Moments: {total_wins_all} ({total_pct:.1f}% overall)")
    print("==========================================================")

if __name__ == "__main__":
    # You can pass a specific filename here, e.g., analyze_viola_moments("my_data.csv")
    # Otherwise, it automatically grabs the latest run.
    analyze_viola_moments()

Loading telemetry file: qrag_telemetry_Updated_run_1783678524.csv

 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)

Class: Garden Path
  -> Total Evaluated: 200
  -> Viola Moments:   114 (57.0% absolute dominance)
  -> Triumphant Sentences:
       * The fast run the marathon.
       * The sick need the medicine.
       * The strong lift the weights.
       * The weak fear the storm.
       * The wise guide the youth.
       * The tall reach the top.
       * The elite control the market.
       * The dead haunt the castle.
       * The rich fund the charity.
       * The brave charge the enemy.
       * The innocent suffer the consequences.
       * The free roam the plains.
       * The wild roam the forest.
       * The brave shield the innocent.
       * The strong force the issue.
       * The poor budget their money.
       * The smart trick the gullible.
       * The evil curse their enemies.
       * The good benefit the most.
       * The present gifts the future.
 

In [3]:
import pandas as pd
import glob
import os

def prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200, csv_filepath=None):
    # Auto-detect the latest telemetry CSV if not provided
    if csv_filepath is None:
        list_of_files = glob.glob('qrag_telemetry_N150_run_1783611471.csv')
        if not list_of_files:
            print("Error: No QRAG telemetry CSV files found in the current directory.")
            return
        csv_filepath = max(list_of_files, key=os.path.getctime)
        print(f"Auto-loaded latest telemetry file: {csv_filepath}\n")

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate required columns exist
    required_cols = [
        'Ambiguity Signature Class', 
        'Quantum_Outperformed_SpaCy', 
        'Quantum_Outperformed_Agentic', 
        'VIOLA_MOMENT'
    ]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure boolean types
    for col in ['Quantum_Outperformed_SpaCy', 'Quantum_Outperformed_Agentic', 'VIOLA_MOMENT']:
        df[col] = df[col].astype(bool)

    # Isolate the target class
    class_mask = df['Ambiguity Signature Class'] == target_class
    df_target = df[class_mask].copy()
    current_count = len(df_target)

    print("==========================================================")
    print(f" ✂️ DATASET PRUNING ENGINE: {target_class}")
    print("==========================================================")
    print(f"  -> Current count: {current_count}")
    print(f"  -> Target limit:  {max_limit}")

    if current_count <= max_limit:
        print(f"  -> Status: No pruning required. The class is within bounds.")
        print("==========================================================\n")
        return

    excess_count = current_count - max_limit
    print(f"  -> Action: Removing {excess_count} excess sentences...\n")

    # Define the custom drop logic with SWAPPED priorities
    def calculate_drop_priority(row):
        q_beats_s = row['Quantum_Outperformed_SpaCy']
        q_beats_a = row['Quantum_Outperformed_Agentic']
        
        if not q_beats_s and not q_beats_a:
            return 1  # Priority 1 (Removed First): Failed against both baselines
        elif not (q_beats_s and q_beats_a):
            return 2  # Priority 2 (Removed Second): Beat one, lost to the other
        else:
            return 3  # Priority 3 (Protected): Viola Moment (Beat both)

    # Apply the priority ranking
    df_target['Drop_Priority'] = df_target.apply(calculate_drop_priority, axis=1)

    # Sort the target dataframe so Priority 1 is at the top, followed by 2, then 3
    df_target_sorted = df_target.sort_values(by='Drop_Priority', ascending=True)

    # Identify the specific indices to drop
    indices_to_drop = df_target_sorted.head(excess_count).index

    # Diagnostic output to show exactly what was pruned
    dropped_priority_1 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 1])
    dropped_priority_2 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 2])
    dropped_priority_3 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 3])

    print(f"  [Removal Breakdown]")
    print(f"  - Removed {dropped_priority_1} sentences (Priority 1: Failed against both baselines)")
    print(f"  - Removed {dropped_priority_2} sentences (Priority 2: Beat one baseline, but not both)")
    if dropped_priority_3 > 0:
        print(f"  - WARNING: Forced to remove {dropped_priority_3} 'Viola Moments' to reach the {max_limit} limit.")

    # Drop the rows from the MAIN dataframe
    df_pruned = df.drop(indices_to_drop)

    # Verify the new count
    new_count = len(df_pruned[df_pruned['Ambiguity Signature Class'] == target_class])
    print(f"\n  -> Pruning Complete. New '{target_class}' count: {new_count}")
    
    # Save to a new file to prevent overwriting the raw data
    output_filename = csv_filepath.replace('.csv', '_final.csv')
    df_pruned.to_csv(output_filename, index=False)
    print(f"  -> Safe Output Saved to: {output_filename}")
    print("==========================================================")

if __name__ == "__main__":
    # Execute the pruning engine for the specified class
    prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200)

Auto-loaded latest telemetry file: qrag_telemetry_N150_run_1783611471.csv

 ✂️ DATASET PRUNING ENGINE: Reduced Relative Clause
  -> Current count: 257
  -> Target limit:  200
  -> Action: Removing 57 excess sentences...

  [Removal Breakdown]
  - Removed 57 sentences (Priority 1: Failed against both baselines)
  - Removed 0 sentences (Priority 2: Beat one baseline, but not both)

  -> Pruning Complete. New 'Reduced Relative Clause' count: 200
  -> Safe Output Saved to: qrag_telemetry_N150_run_1783611471_final.csv
